# From Script to Testable Python

Lecture 05A uses a small Codex-generated Python tool as the working example.

The review workflow: inspect the context, run the program, probe the functions, run tests, run tool checks, and decide what to ask Codex next.

The main ideas are practical:

- pure functions are easier to test because they take inputs, return outputs, and avoid side effects
- `main()` and the main guard keep import behavior clean
- type hints make expectations visible even though Python remains dynamic
- `pytest` gives us repeatable checks for small pieces of behavior
- Ruff, mypy, and Git help us review before committing

## Setup Assumptions

Assumptions:

- you cloned the `testable-python-lab` repo
- you started Jupyter from the repo root
- you have not necessarily installed `pytest`, Ruff, or mypy yet

The repo already contains a refactored `tiny_data_generator.py` and a first test file. We are not building them from scratch today. We are inspecting a fixed Codex-style result so everyone can discuss the same code.

## Notebook Working Directory

Jupyter may start the kernel in the repo root or in the `notebooks/` folder. This setup cell normalizes the notebook to the repo root so the shell commands below match the repo layout.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

## The Prompt We Assume

Assume we already asked Codex for a first version:

```text
Build a tiny Python script that prints fake CSV data. I want to be able to choose how many rows it prints. Please include tests.
```

Codex can often produce a useful first version from a prompt like this. The review question is whether the result has a structure we can run, import, test, and keep improving.

## Inspect The Repo

Start with shell inspection. Use the terminal tools directly so you see the project shape Codex was working inside.

Note: Jupyter can run shell commands. Just use a code cell and prepend the command with a `!`.

In [ ]:
!pwd
!fd --type f --max-depth 3

Expected core files:

```text
./AGENTS.md
./README.md
./tiny_data_generator.py
./tests/test_tiny_data_generator.py
./notebooks/05a-testable-python.ipynb
./pyproject.toml
```

If you do not see that shape, stop and restart Jupyter from the cloned repo.

## Read `AGENTS.md`

`AGENTS.md` gives Codex repo-level context before it starts changing files.

That matters here because this repo is not just a normal tiny Python project. It is a teaching repo. We want Codex to preserve that premise, keep changes small, avoid unnecessary dependencies, and explain work in a way students can inspect.

In [ ]:
!cat AGENTS.md

Practical questions:

- What does `AGENTS.md` tell Codex that is not obvious from the file names?
- What constraints would make Codex less likely to overbuild?
- What should Codex explain before editing this repo?

## Read The Repo README

In [ ]:
!cat README.md

The README gives human context. `AGENTS.md` gives agent context. They overlap, but they are not the same audience.

## Inspect The Generated Script

Read the script before running it.

In [ ]:
!cat tiny_data_generator.py

Before running anything, identify:

- the functions Codex created
- which function generates data
- which function formats data
- which function coordinates the command-line program
- where the main guard appears
- which lines have type hints

## Python File Structure

A Python file can be used in two ways:

- run directly as a script
- imported as a module by tests, notebooks, or other Python files

A useful script usually separates reusable code from command-line behavior.

```python
def useful_function(input_value):
    return output_value


def main():
    result = useful_function(input_value)
    print(result)


if __name__ == "__main__":
    main()
```

The reusable functions can be imported and tested. `main()` coordinates the script behavior. The main guard makes sure the script runs only when the file is executed directly.

## Install The Review Tools

The slides only installed Jupyter. For this lab, add the development tools from inside the project.

In [ ]:
!uv add --dev pytest ruff mypy

This changes the project environment and may update `pyproject.toml` and `uv.lock`. That is expected in a real repo, and it is one reason we inspect `git diff` before committing.

## Run The Tool

Run the script from the shell.

In [ ]:
!uv run python tiny_data_generator.py

Try the command-line option.

In [ ]:
!uv run python tiny_data_generator.py --rows 3

This is useful, but it is not enough. A command-line run proves one visible example worked. It does not give us small, repeatable checks for the parts of the code.

## Pure Functions, Practically

A pure function has two properties:

- the same input always produces the same output
- the function has no side effects

A side effect is a change or interaction outside the function's return value: printing, writing a file, reading user input, changing global state, calling an API, or depending on the current time.

Pure function:

```python
def add_one(number: int) -> int:
    return number + 1
```

Not pure:

```python
def print_one_more(number: int) -> None:
    print(number + 1)
```

The second function may be useful, but it is harder to test because the result is printed instead of returned.

Design goal: keep the core logic pure when possible and keep side effects near `main()`.

## Import The Core Functions

Tests and notebooks can import functions without running the command-line program.

In [ ]:
from tiny_data_generator import format_csv, generate_rows

If importing printed a CSV table, that would be a design problem. Importing should make names available; it should not run the whole program. This is the point of if-name-main.

## Check `generate_rows`

In [ ]:
generate_rows(3)

What makes this testable?

- the input is a plain integer
- the output is a plain Python list
- the rows are dictionaries with predictable keys
- there is no terminal output to capture
- there is no file path or current-folder assumption

`generate_rows` and `format_csv` are the testable core: inputs in, outputs out, no printing, no files.

`main()` is the boundary: it deals with command-line arguments and terminal output.

Try a boundary value.

In [ ]:
generate_rows(0)

Should zero rows be allowed? Should negative rows be rejected? The current code has an answer, but the important engineering habit is this: once you decide the rule, write a test for it.

## Check `format_csv`

In [ ]:
rows = generate_rows(2)
csv_text = format_csv(rows)
print(csv_text)

Because `format_csv` returns a string, we can inspect it directly.

In [ ]:
assert csv_text.startswith("id,name,score")
assert "1,user_1,71" in csv_text
assert "2,user_2,72" in csv_text

Those `assert` lines are not a replacement for a test file, but they show the shape of a unit test: call a small function, compare actual behavior to expected behavior.

## `main()` And The Main Guard

Now look back at the bottom of the script.

In [ ]:
!tail -n 40 tiny_data_generator.py

`main()` is where the command-line program gets coordinated:

- parse the command-line arguments
- call the data-generation function
- call the CSV-formatting function
- print the final text

When the file is run directly:

`uv run python tiny_data_generator.py`

Python sets the file's special `__name__` value to `"__main__"`, so the guarded `main()` call runs.

When another file imports from it:

`from tiny_data_generator import generate_rows`

Python gives the module a normal import name, so the guarded `main()` call does not run.

## Type Hints As Lightweight Documentation

Python is dynamically typed: the type lives with the value at runtime.

```python
value = 10
value = "ten"
```

Many statically typed languages attach the type more directly to the variable, so a variable declared as a number cannot later hold text.

Python's flexibility is useful while exploring. The cost is that type mistakes can appear later as runtime bugs.

Type hints let us write expected types in the code:

```python
def generate_rows(count: int) -> list[dict[str, str]]:
    ...
```

Important distinction:

- dynamic typing: how Python runs
- type hints: expected types written in the code
- static checking: tools check those expectations before runtime

Type hints usually do not change runtime behavior. They make code easier for people, editors, Codex, and type checkers to understand.

Type hints answer review questions quickly:

- What kind of value should `generate_rows` receive?
- What kind of value should it return?
- Does `format_csv` return text or print text?
- Does `main()` return a useful value?

Inspect just the function signatures.

In [ ]:
!rg '^def ' tiny_data_generator.py

Useful reading:

- `count: int` says callers should pass an integer
- `list[dict[str, str]]` says the result is a list of row dictionaries
- `-> str` says the function returns text
- `-> None` says the function is called for its effect, not its return value

The type hints do not prove the code is correct. They make the intended contract visible to humans, Codex, editors, and mypy.

## Inspect The Tests

Read the tests before running them.

In [ ]:
!cat tests/test_tiny_data_generator.py

## Unit Tests

A unit test checks one small behavior directly.

The test should:

- call a function
- provide known input
- compare actual output to expected output

Unit tests are easiest when the code has pure or mostly pure functions.

Other tests exist. Integration tests check whether pieces work together. End-to-end tests check a full user path. Unit tests are the smallest starting point.

For each test, ask:

- what function does it call?
- what input does it use?
- what expected behavior does it name?
- what behavior is still not checked?

## Run The Tests

In [ ]:
!uv run python -m pytest

This form is a little more explicit than `uv run pytest`.

Because this teaching repo keeps `tiny_data_generator.py` as a flat script at the repo root, running pytest through the project Python makes the import path behave the way we want.

Passing tests mean the checked behaviors still work. They do not prove the whole program is correct.

The point is narrower and more useful: tests give Codex and humans a repeatable target while the code changes.

## Run The Tool Checks

Different tools answer different questions:

- `ruff format --check .`: would Ruff reformat any files?
- `ruff check .`: does Ruff see style, import, or bug-risk issues?
- `mypy tiny_data_generator.py`: do the type hints line up with the code?
- `python -m pytest`: do the tested behaviors still pass?

In [ ]:
!uv run ruff format --check .
!uv run ruff check .
!uv run mypy tiny_data_generator.py
!uv run python -m pytest

These tools do not decide whether the program is useful. They catch routine issues before higher-level review.

## Ask Codex To Explain Before Editing

Use Codex as a reviewer before using it as an editor.

```text
Read AGENTS.md, tiny_data_generator.py, and tests/test_tiny_data_generator.py. Explain the code structure in terms of pure functions, boundary code, type hints, and tests. Do not edit files.
```

Another useful review prompt:

```text
Do not edit yet. First tell me which functions are easiest to test, which code is boundary code, and what one small test you would add next.
```

Then ask for coverage review:

```text
Review tests/test_tiny_data_generator.py. What important behavior is checked, what behavior is not checked, and what is one small additional test you would add next? Do not edit files.
```

## A Small Next Change

If we decide negative row counts should be rejected, ask for one focused edit:

```text
Update the command-line behavior so negative row counts are rejected with a clear error. Keep zero rows allowed. Add or update pytest tests for this behavior. Keep the change small and explain the diff before committing.
```

Notice the shape of that request:

- one behavior change
- one rule stated clearly
- tests required
- explanation required before commit

## Inspect The Diff

Before committing, inspect what changed.

In [ ]:
!git status
!git diff --stat

If there is a detailed diff to review:

In [ ]:
!git diff

## Takeaway

The goal is not Python from scratch.

The goal is code supervision:

- use `AGENTS.md` to give Codex useful context
- inspect generated files before trusting them
- keep core logic in small functions with clear inputs and outputs
- use `main()` and the main guard for clean script behavior
- read type hints as practical contracts
- use `pytest`, Ruff, mypy, and Git diff before committing